In [3]:
import pandas as pd #para la lectura de datos
import redis

# Conexión a Redis local
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
#r = redis.Redis(host='157.92.26.78', port=6379, decode_responses=True)
# Decodifica automáticamente las respuestas de Redis para que no devuelva siempre bytes.

# Probar la conexión
r.ping()

True

### Modelo 1: Estado del turno

In [55]:
# Cargamos los datos
df_turnos = pd.read_csv("turno",header=None)

In [56]:
for _, row in df_turnos.iterrows(): # por cada fila del df de turnos
    r.set(                          # creamos una clave valor (turno,estado)
        f"estado_turno:{row[0]}:estado",
        row[5]
    )

In [57]:
# Ahora supongamos que una persona quiere saber si un turno no ha sido cancelado:
id_turno = 4
print('El estado del turno', id_turno, 'es:', r.get(f"estado_turno:{4}:estado"))


El estado del turno 4 es: realizado


In [58]:
# Una persona cancelo un turno:
r.set(f"estado_turno:{id_turno}:estado",'cancelado')
print('El estado del turno', id_turno, 'es:', r.get(f"estado_turno:{4}:estado"))

El estado del turno 4 es: cancelado


### Modelo 2: Perfil de un paciente.

In [82]:
# Cargamos los datos
df_paciente = pd.read_csv("paciente")
#estos datos fueron tomados de una consulta en la que se joineo pacientes, persona y obra social.

In [83]:
print(df_paciente.columns)

Index(['cuil', 'nombre', 'apellido', 'genero', 'grupo_sanguineo',
       'fecha_nacimiento', 'telefono', 'mail', 'nombre_obra'],
      dtype='object')


In [106]:
for _, row in df_paciente.iterrows(): # por cada fila del df de pacientes
    r.hset(                           # armo un hash con el cuil como clave y me guardo todos los datos.
        f"paciente:{row['cuil']}",
        mapping={
            "nombre": row['nombre'],
            "apellido": row['apellido'],
            "genero": row['genero'],
            "grupo_sanguineo": row['grupo_sanguineo'],
            "fecha_nacimiento": row['fecha_nacimiento'],
            "telefono": row['telefono'],
            "mail": row['mail'],
            "nombre_obra": row['nombre_obra']
        }
    )

In [107]:
# supongamos que quiero saber el grupo sanguineo del paciente con el siguiente cuil:
cuil = 1234567893
print("El grupo sanguineo del paciente con cuil", cuil, "es", r.hget(f'paciente:{cuil}',"grupo_sanguineo"))

El grupo sanguineo del paciente con cuil 1234567893 es A+


In [108]:
# si quiero todos los datos del mismo paciente:
print(r.hgetall(f'paciente:{cuil}'))

{'nombre': 'nombre4', 'apellido': 'apellido4', 'genero': 'hombre', 'grupo_sanguineo': 'A+', 'fecha_nacimiento': '1999-11-10', 'telefono': '1123451237', 'mail': 'mail4@mail.cm', 'nombre_obra': 'galeno'}


In [109]:
# el paciente cambio su mail
print("El mail del paciente con cuil", cuil, "antes de actualizarlo es", r.hget(f'paciente:{cuil}',"mail"))
r.hset(f"paciente:{cuil}",mapping = {"mail": "nuevo@mail.com"})
print("El mail del paciente con cuil", cuil, "después de actualizarlo es", r.hget(f'paciente:{cuil}',"mail"))


El mail del paciente con cuil 1234567893 antes de actualizarlo es mail4@mail.cm
El mail del paciente con cuil 1234567893 después de actualizarlo es nuevo@mail.com
